# **1. Q-Learning para _cliff walking_**

En este cuaderno, vamos a resolver el problema de [Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) con **Q-Learning**, un algoritmo de aprendizaje por refuerzo que aprende el valor $Q(s, a)$ de cada par estado-acción y lo almacena en una tabla.

_Considere el mundo de cuadrícula ("gridworld") mostrado abajo. Es una tarea episódica estándar, con un estado inicial (casilla S) y meta (casilla G). Las acciones posibles para moverse son arriba, abajo, derecha e izquierda. La recompensa es -1 en todas las transiciones, salvo en las que llevan a la región marcada como «El Acantilado» ("The Cliff"). Pisar esta región supone una recompensa de -100 y devuelve al agente instantáneamente al estado inicial._

<img src="../../img/cliffwalking.png" width=500/>

Basado en:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Capítulos 6.5: Q-learning: control TD _off-policy_, 6.6: _Expected Sarsa_.

> Udacity (2019) Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/blob/master/temporal-difference/Temporal_Difference_Solution.ipynb).

## **1.0. Preparación**

Comenzamos importando las bibliotecas necesarias, así como otras utilidades que emplearemos más adelante:

In [ ]:
import os
import gymnasium as gym
import numpy as np
import random
from collections import deque

import io
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
from PIL import Image

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def show_episode(env, act):
    """
    Ejecuta un episodio completo y lo muestra como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción a realizar en un estado
    """
    state, info = env.reset()
    frames = [Image.fromarray(env.render())]
    episode_reward = 0
    terminated = truncated = False

    # Bucle de interacción agente-entorno (1 episodio)
    while not (terminated or truncated):
        action = act(state)
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(Image.fromarray(env.render()))
        episode_reward += reward

    # Guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format="gif", save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata["render_fps"], loop=0)
    display(IPImage(data=gif.getvalue(), format="gif"))
    print("Recompensa del episodio: ", episode_reward)

def plot_rewards(episode_rewards, title, solved_reward=None, window=100):
    """
    Representa la recompensa de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        episode_rewards (list): recompensa de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_reward (float): recompensa media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    episode_rewards = np.asarray(episode_rewards)
    avg = [np.mean(episode_rewards[max(0, i-window+1):i+1]) for i in range(len(episode_rewards))]
    plt.plot(np.arange(len(episode_rewards)), episode_rewards, alpha=0.4, label="Recompensa")
    plt.plot(np.arange(len(episode_rewards)), avg, label=f"Recompensa media (últimos {window})")
    if solved_reward is not None:
        plt.axhline(solved_reward, color="gray", linestyle="--", label="Resuelto")
    plt.ylabel("Recompensa")
    plt.xlabel("Episodio nº")
    plt.title(title)
    plt.legend()
    plt.show()

def print_policy(actions):
    """
    Muestra la política como una cuadrícula de flechas (C: acantilado, T: meta).

    Parámetros
    ==========
        actions (array_like): acción de cada uno de los 48 estados (ARRIBA = 0, DERECHA = 1, ABAJO = 2, IZQUIERDA = 3)
    """
    grid = np.array(["↑", "→", "↓", "←"])[np.asarray(actions)].reshape(4, 12)
    grid[3, 1:11] = "C"   # Celdas del acantilado
    grid[3, 11] = "T"     # Meta
    print("\n".join(" ".join(row) for row in grid))

def plot_values(V):
    """
    Representa la función de valor de estado sobre la cuadrícula de 4 x 12 celdas.

    Parámetros
    ==========
        V (array_like): valor de cada uno de los 48 estados
    """
    V = np.reshape(V, (4, 12))
    fig = plt.figure(figsize=(15, 5))
    ax = fig.add_subplot(111)
    ax.imshow(V, cmap="cool")
    for (j, i), label in np.ndenumerate(V):
        ax.text(i, j, f"{label:.1f}", ha="center", va="center", fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title("Función de valor de estado")
    plt.show()

## **1.1. Entorno**

[Gymnasium](https://gymnasium.farama.org/), antes conocido como Open AI Gym, es un estándar de API para el aprendizaje por refuerzo con una amplia colección de entornos de referencia, entre ellos [Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/).

Podemos crear una instancia del entorno, el objeto [`env`](https://gymnasium.farama.org/api/env/#gymnasium-env), e interactuar con él mediante una interfaz común que implementa las operaciones básicas del bucle agente-entorno.

<img src="../../img/rl-loop.png" width=300/>

Las funciones principales son las siguientes (según la [documentación](https://gymnasium.farama.org/api/env/#) de la API de Gymnasium):

*   [`reset`](https://gymnasium.farama.org/api/env/#gymnasium.Env.reset): Reinicia el entorno a un estado inicial.  Devuelve el estado inicial del episodio e información adicional (métricas, información de depuración). Es necesario llamarla antes del primer `step`.
*   [`step`](https://gymnasium.farama.org/api/env/#gymnasium.Env.step): Aplica una acción sobre el entorno y devuelve varios valores: el nuevo estado, la recompensa obtenida, si el episodio ha terminado (_terminated_) o se ha truncado (_truncated_) e información adicional sobre el paso (métricas, información de depuración).
*   [`render`](https://gymnasium.farama.org/api/env/#gymnasium.Env.render): Genera una imagen del entorno para ayudar a visualizar lo que ve el agente.
*   [`close`](https://gymnasium.farama.org/api/env/#gymnasium.Env.close): Cierra el entorno.

Caer por el acantilado no termina el episodio. Para evitar que un episodio nunca finalice, limitamos la longitud de los episodios a $100$ pasos con el parámetro `max_episode_steps`. Al alcanzar ese límite, el episodio se trunca (`truncated = True`).

In [ ]:
env = gym.make("CliffWalking-v1", render_mode="rgb_array", max_episode_steps=100)

print(env.observation_space)
print(env.action_space)

El estado es el índice de la celda en la que se encuentra el agente (de $0$ a $47$) y hay $4$ acciones posibles: `0` (arriba), `1` (derecha), `2` (abajo) y `3` (izquierda).

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

A continuación, se muestra el bucle de interacción agente-entorno para un agente aleatorio, que elige cada acción al azar del espacio de acciones ([`action_space`](https://gymnasium.farama.org/api/spaces/#gymnasium.spaces.Space.sample)). El episodio acaba cuando el agente llega a la meta (terminación) o cuando se alcanza el límite de pasos (truncado).

In [ ]:
env.reset()
terminated = truncated = False

# Bucle de interacción agente-entorno (1 episodio)
while not (terminated or truncated):
    action = env.action_space.sample() # Devuelve una acción aleatoria del espacio de acciones
    state, reward, terminated, truncated, info = env.step(action)

    # Mostrar imagen del entorno
    show_state(env)

print("¿Episodio terminado? ", terminated)
print("¿Episodio truncado? ", truncated)

La función `show_episode()` ejecuta este mismo bucle y muestra el episodio como una animación, junto con la **recompensa del episodio**, es decir, la suma de las recompensas obtenidas en cada paso. La usaremos para comparar el comportamiento del agente antes y después del entrenamiento:

In [ ]:
show_episode(env, lambda state: env.action_space.sample())

## **1.2. Algoritmo Q-Learning**

A continuación, se muestra una implementación de Q-Learning con la misma estructura que usaremos en los siguientes cuadernos para DQN y REINFORCE:

*   `Agent`: Implementación de las capacidades del agente: elegir una acción con la política $\epsilon$-_greedy_ (`act`) y actualizar la tabla $Q$ con cada experiencia (`step`).
*   `qlearning()`: Bucle de entrenamiento.

Recordemos la regla de actualización de la tabla $Q$:

$Q(S_t, A_t) ← Q(S_t, A_t) + α·[R_{t+1} + γ \max_{a}Q(S_{t+1}, a) - Q(S_t,A_t)]$

En `Agent.step()` la actualización se hace en dos pasos:

1.  **Objetivo** (`Q_target`): lo que esta experiencia sugiere que vale $Q(S_t, A_t)$, es decir, la recompensa obtenida más el mejor valor estimado desde el estado siguiente, $R_{t+1} + γ \max_{a}Q(S_{t+1}, a)$.
2.  **Actualización**: $Q(S_t, A_t)$ se acerca a ese objetivo una fracción $α$ de la diferencia.

El factor `(1 - done)` anula el segundo término cuando $S_{t+1}$ es terminal. Un estado terminal no tiene futuro, así que su valor es $0$ por definición, y el objetivo se reduce a la recompensa. Solo `terminated` cuenta como terminal. Si el episodio se trunca por el límite de pasos, el estado siguiente no es terminal y se sigue estimando su valor.

En _cliff walking_, la fila de la meta en la tabla $Q$ nunca se actualiza y vale $0$ de todas formas, así que aquí `(1 - done)` no cambia el resultado. Lo incluimos porque es la forma correcta en general y porque en DQN sí será necesario: la red neuronal puede devolver valores distintos de $0$ para un estado terminal.

In [ ]:
GAMMA = 1.0             # Factor de descuento (gamma)
ALPHA = 0.01            # Tasa de aprendizaje (alpha)

class Agent():
    """Interactúa con el entorno y aprende de él."""

    def __init__(self, state_size, action_size, seed):
        """Inicializa un objeto Agent.

        Parámetros
        ==========
            state_size (int): número de estados
            action_size (int): número de acciones
            seed (int): semilla aleatoria
        """
        self.state_size = state_size
        self.action_size = action_size
        random.seed(seed)

        # Tabla Q, inicializada a ceros
        self.Q = np.zeros((state_size, action_size))

    def act(self, state, eps=0.):
        """Devuelve la acción para el estado dado según la política actual.

        Parámetros
        ==========
            state (int): estado actual
            eps (float): epsilon, para la selección de acciones ε-greedy
        """
        # Selección de acciones ε-greedy
        if random.random() > eps:
            return np.argmax(self.Q[state])
        else:
            return random.choice(np.arange(self.action_size))

    def step(self, state, action, reward, next_state, done):
        """Actualiza la tabla Q con una experiencia.

        Parámetros
        ==========
            state (int): estado actual (St)
            action (int): acción tomada (At)
            reward (float): recompensa obtenida (Rt+1)
            next_state (int): siguiente estado (St+1)
            done (bool): si St+1 es terminal (no se hace bootstrapping desde él)
        """
        # Calcular el objetivo: Rt+1 + gamma * max_a Q(St+1, a)
        Q_target = reward + GAMMA * np.max(self.Q[next_state]) * (1 - done)

        # Acercar Q(St, At) al objetivo
        self.Q[state, action] += ALPHA * (Q_target - self.Q[state, action])

Durante todo el entrenamiento, el agente sigue una política $\epsilon$-_greedy_ con $\epsilon = 0.2$ constante. Por ello, aunque aprenda la política óptima (que obtiene una recompensa de $-13$), algunas acciones aleatorias le seguirán haciendo caer por el acantilado, y la recompensa media durante el entrenamiento será menor.

In [ ]:
def qlearning(agent, n_episodes=25000, eps=0.2, print_every=1000):
    """Q-Learning.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número de episodios
        eps (float): valor de epsilon (constante)
        print_every (int): mostrar la recompensa media cada print_every episodios
    """
    episode_rewards = []                # Recompensas de todos los episodios
    rewards_window = deque(maxlen=100)  # Recompensas de los últimos 100 episodios

    for i_episode in range(1, n_episodes+1):
        state, info = env.reset()
        episode_reward = 0
        terminated = truncated = False

        # Bucle de interacción agente-entorno (1 episodio)
        while not (terminated or truncated):
            # Elegir la acción At con la política ε-greedy
            action = agent.act(state, eps)

            # Aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, info = env.step(action)

            # Actualizar la tabla Q con <St, At, Rt+1, St+1>
            # (solo 'terminated' marca St+1 como terminal;
            #  tras un truncamiento hay que seguir haciendo bootstrapping)
            agent.step(state, action, reward, next_state, terminated)

            # Avanzar al siguiente estado
            state = next_state
            episode_reward += reward

        rewards_window.append(episode_reward)   # Guardar la recompensa del episodio (ventana de 100)
        episode_rewards.append(episode_reward)  # Guardar la recompensa del episodio (todos los episodios)

        # Mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print(f"Episodio {i_episode}\tRecompensa media (últimos 100): {np.mean(rewards_window):.2f}")

    os.makedirs("agentes", exist_ok=True)
    np.save("agentes/cliffwalking-qlearning.npy", agent.Q) # Guardar la tabla Q del agente entrenado

    return episode_rewards

agent = Agent(state_size=48, action_size=4, seed=0)
episode_rewards = qlearning(agent)

plot_rewards(episode_rewards, "Q-Learning")

## **1.3. Política aprendida y función de valor de estado**

La política aprendida (_greedy_) elige en cada estado la acción con mayor valor en la tabla $Q$:

In [ ]:
print_policy(agent.Q.argmax(axis=1))

Podemos visualizar los valores de la función de valor de estado $v_\pi$ para la política estimada $\pi$ y compararlos con los valores de la política óptima $v_{\pi_*}$.

__Función de valor de estado de la política aprendida por el agente $v_\pi$__

In [ ]:
V = agent.Q.max(axis=1)
plot_values(V)

__Función de valor de estado de la política óptima $v_{\pi_*}$__

In [ ]:
# Los valores de estado de la política óptima pueden calcularse manualmente
V_opt = np.zeros((4, 12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## **1.4. Visualizar el agente entrenado**

Cargamos la tabla $Q$ en el agente y lo ejecutamos durante un episodio con la política _greedy_:

In [ ]:
# Cargar la tabla Q desde `agentes/cliffwalking-qlearning.npy`
agent_load = Agent(state_size=48, action_size=4, seed=0)

agent_load.Q = np.load("agentes/cliffwalking-qlearning.npy")

show_episode(env, agent_load.act)

¡Nuestro agente ya sabe cruzar el acantilado por el camino más corto!

En este cuaderno, hemos aprendido a interactuar con un entorno de Gymnasium y a resolverlo con Q-Learning, almacenando el valor de cada par estado-acción en una tabla. También hemos visualizado la política aprendida y su función de valor de estado.

En el próximo cuaderno, sustituiremos la tabla $Q$ por una red neuronal (Deep Q-Learning), lo que permite abordar problemas con estados continuos, como CartPole.